# Notebook 1: Data Collection and Feature Engineering

This notebook downloads and assembles the core dataset used throughout the project. We pull daily VIX closing levels from Yahoo Finance and four Central and Eastern European (CEE) equity index series from Stooq and the Vienna Stock Exchange. The output is a single regression-ready parquet file that all subsequent notebooks consume.

**Outputs:**
- `data/prices_aligned.parquet`: aligned daily closing prices, all five series
- `data/cee_returns.parquet`: log daily returns for the four CEE indices
- `data/vix.parquet`: VIX level series
- `data/regression_data.parquet`: main analysis file (VIX features + next-day CEE returns)
- `figures/01_raw_series.png`

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

Path("../data").mkdir(exist_ok=True)
Path("../figures").mkdir(exist_ok=True)

print("Packages loaded.")

All packages loaded.


In [ ]:
tickers = {
    "VIX": "^VIX"
}

start_date = "2018-01-01"
end_date   = "2024-12-31"

raw = yf.download (
    tickers     = list(tickers.values()),
    start       = start_date,
    end         = end_date,
    auto_adjust = True,
    progress    = True
)

# yfinance returns a different structure when only one ticker is requested
prices = raw[["Close"]].copy()
prices.columns = ["VIX"]

# add placeholder columns for the four CEE indices
prices["WIG20"] = None
prices["BUX"]   = None
prices["PX"]    = None
prices["ATX"]   = None

print(f"Rows downloaded: {len(prices)}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nColumns: {list(prices.columns)}")
prices.tail()

# strip out weekends from ATX
atx = atx[atx.index.dayofweek < 5]
prices["ATX"] = atx

print(f"ATX rows after weekday filter: {len(atx)}")
print(f"\nUpdated missing values per column:")
print(prices.isna().sum())

[*********************100%***********************]  1 of 1 completed

Rows downloaded: 1760
Date range: 2018-01-02 to 2024-12-30

Columns: ['VIX', 'WIG20', 'BUX', 'PX', 'ATX']
ATX rows after weekday filter: 1789

Updated missing values per column:
VIX         0
WIG20    1760
BUX      1760
PX       1760
ATX        27
dtype: int64


In [16]:
def load_stooq(path, start, end):
    df = (
        pd.read_csv(path, parse_dates=["Date"])
        .sort_values("Date")
        .set_index("Date")
        .loc[start:end, "Close"]
    )
    return df

def load_wienerborse(path, start, end):
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    df = df.dropna(axis=1, how="all")
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    df = df.sort_values("Date").set_index("Date")
    df = df.loc[start:end, "Last Close"]
    # filter out weekends
    df = df[df.index.dayofweek < 5]
    return df

wig20 = load_stooq("../data/wig20_stooq.csv", start_date, end_date)
bux   = load_stooq("../data/bux_stooq.csv",   start_date, end_date)
px    = load_stooq("../data/px_stooq.csv",    start_date, end_date)
atx   = load_wienerborse("../data/atx_wienerborse.csv", start_date, end_date)

prices["WIG20"] = wig20
prices["BUX"]   = bux
prices["PX"]    = px
prices["ATX"]   = atx

print(f"WIG20: {len(wig20)} rows")
print(f"BUX:   {len(bux)} rows")
print(f"PX:    {len(px)} rows")
print(f"ATX:   {len(atx)} rows")
print(f"\nMissing values per column:")
print(prices.isna().sum())
prices.tail()

WIG20: 1748 rows
BUX:   1743 rows
PX:    1753 rows
ATX:   1789 rows

Missing values per column:
VIX       0
WIG20    59
BUX      63
PX       53
ATX      27
dtype: int64


,VIX,WIG20,BUX,PX,ATX
Date,,,,,
2024-12-23,16.780001,2202.17,79492.88,1761.74,3600.79
2024-12-24,14.270000,NaN,NaN,NaN,3595.93
2024-12-26,14.730000,NaN,NaN,NaN,NaN
2024-12-27,15.950000,2204.19,NaN,1761.01,3628.99
2024-12-30,17.400000,2192.01,79326.66,1760.17,3647.25


In [ ]:
# inner join: keep only dates where ALL five series have a value
prices_aligned = prices.dropna(how="any")

dropped = len(prices) - len(prices_aligned)
print(f"Before alignment : {len(prices):>5} rows")
print(f"After alignment  : {len(prices_aligned):>5} rows  ({dropped} dates dropped)")
print(f"Date range       : {prices_aligned.index[0].date()} to {prices_aligned.index[-1].date()}")

prices_aligned.to_parquet("../data/prices_aligned.parquet")
prices_aligned.tail()

Before alignment :  1760 rows
After alignment  :  1637 rows  (123 dates dropped)
Date range       : 2018-01-03 to 2024-12-30

Saved → data/prices_aligned.parquet


,VIX,WIG20,BUX,PX,ATX
Date,,,,,
2024-12-18,27.620001,2226.58,79243.31,1754.20,3599.81
2024-12-19,24.090000,2225.51,78665.14,1762.33,3590.45
2024-12-20,18.360001,2200.39,78741.84,1754.62,3581.40
2024-12-23,16.780001,2202.17,79492.88,1761.74,3600.79
2024-12-30,17.400000,2192.01,79326.66,1760.17,3647.25


In [ ]:
cee_cols = ["WIG20", "BUX", "PX", "ATX"]

# log returns for the four equity indices
cee_returns = np.log(prices_aligned[cee_cols]).diff()

# VIX: level (today) and daily change (today's level - yesterday's)
vix_level  = prices_aligned["VIX"]
vix_change = prices_aligned["VIX"].diff()

# build master regression dataframe
# for each date t: today's VIX level, today's VIX change, tomorrow's CEE return
df = pd.DataFrame({
    "VIX_level"  : vix_level,
    "VIX_change" : vix_change,
}, index=prices_aligned.index)

for col in cee_cols:
    df[f"{col}_next"] = cee_returns[col].shift(-1)

df = df.dropna()

print(f"Regression-ready rows: {len(df)}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Regression-ready rows: 1635
Date range: 2018-01-04 to 2024-12-23

Columns: ['VIX_level', 'VIX_change', 'WIG20_next', 'BUX_next', 'PX_next', 'ATX_next']


,VIX_level,VIX_change,WIG20_next,BUX_next,PX_next,ATX_next
Date,,,,,,
2018-01-04,9.22,0.070001,-0.001430,0.003636,0.000172,0.000124
2018-01-05,9.22,0.000000,0.007507,0.002718,0.003252,-0.000183
2018-01-08,9.52,0.300000,-0.007969,-0.005864,-0.004012,0.002336
2018-01-09,10.08,0.559999,-0.006790,-0.005491,-0.002493,0.009308
2018-01-10,9.82,-0.260000,0.009231,0.003060,0.005386,0.002765


In [21]:
# save the master regression dataset and intermediate files
df.to_parquet("../data/regression_data.parquet")
cee_returns.dropna().to_parquet("../data/cee_returns.parquet")
prices_aligned[["VIX"]].to_parquet("../data/vix.parquet")

print("Saved:")
print("  ../data/prices_aligned.parquet   ← raw aligned prices, 5 columns")
print("  ../data/cee_returns.parquet      ← log returns, 4 markets")
print("  ../data/vix.parquet              ← VIX level series")
print("  ../data/regression_data.parquet  ← MAIN FILE for all subsequent notebooks")

print("\n" + "="*55)
print("File check:\n")

files = [
    "../data/prices_aligned.parquet",
    "../data/cee_returns.parquet",
    "../data/vix.parquet",
    "../data/regression_data.parquet",
]
for path in files:
    d = pd.read_parquet(path)
    print(f"  {path}")
    print(f"    {d.shape[0]} rows × {d.shape[1]} cols | "
          f"{d.index[0].date()} → {d.index[-1].date()}\n")

Saved:
  ../data/prices_aligned.parquet   ← raw aligned prices, 5 columns
  ../data/cee_returns.parquet      ← log returns, 4 markets
  ../data/vix.parquet              ← VIX level series
  ../data/regression_data.parquet  ← MAIN FILE for all subsequent notebooks

File check:

  ../data/prices_aligned.parquet
    1637 rows × 5 cols | 2018-01-03 → 2024-12-30

  ../data/cee_returns.parquet
    1636 rows × 4 cols | 2018-01-04 → 2024-12-30

  ../data/vix.parquet
    1637 rows × 1 cols | 2018-01-03 → 2024-12-30

  ../data/regression_data.parquet
    1635 rows × 6 cols | 2018-01-04 → 2024-12-23



In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 10))
axes = axes.flatten()

# VIX level
axes[0].plot(prices_aligned.index, prices_aligned["VIX"],
             color="#e24b4a", linewidth=0.7)
axes[0].set_title("VIX level", fontsize=12, fontweight="500")
axes[0].axhline(25, color="gray", linewidth=0.5,
                linestyle="--", label="VIX = 25 (crisis threshold)")
axes[0].legend(fontsize=9)

# four CEE return series
colors = {"WIG20": "#2563eb", "BUX": "#7c3aed",
          "PX": "#059669", "ATX": "#d97706"}

for i, col in enumerate(cee_cols, start=1):
    axes[i].plot(cee_returns.index, cee_returns[col],
                 color=colors[col], linewidth=0.5, alpha=0.85)
    axes[i].set_title(f"{col} log returns", fontsize=12, fontweight="500")
    axes[i].axhline(0, color="gray", linewidth=0.4, linestyle="--")

# hide unused 6th panel
axes[5].set_visible(False)

for ax in axes[:5]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(labelsize=9)

fig.suptitle("VIX and CEE daily log returns, 2018 to 2024",
             fontsize=14, fontweight="500", y=1.01)
plt.tight_layout()
plt.savefig("../figures/01_raw_series.png", dpi=150, bbox_inches="tight")
plt.show()